# Reproducing BacteReason: Distillation & Inference of a Scientific Reasoning Model

---
## 0. Install Dependencies

In [ ]:
%%capture
!pip install unsloth huggingface_hub
!pip install --no-deps xformers trl peft accelerate bitsandbytes

---
## 1. GPU Detection & Model Selection

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found. Go to Runtime > Change runtime type > GPU."

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:  {GPU_NAME}")
print(f"VRAM: {VRAM_GB:.1f} GB")

if VRAM_GB >= 40:
    BASE_MODEL     = "Qwen/QwQ-32B"
    MAX_SEQ_LENGTH = 32768
    print("\n-> Using QwQ-32B — paper configuration (A100)")
else:
    BASE_MODEL     = "Qwen/Qwen2.5-3B-Instruct"
    MAX_SEQ_LENGTH = 4096
    print("\n-> Using Qwen2.5-3B-Instruct — T4 proxy")
    print("   Workflow is identical; results will differ from the paper.")
    print("   For full reproduction, use an A100 GPU (Colab Pro).")

---
## 2. Download & Load Data

Currently limited data

In [ ]:
from huggingface_hub import hf_hub_download
import json

REPO_ID = "Playingyoyo/sotsuken_rotation_2026"

TRAIN_DATA_PATH = hf_hub_download(
    repo_id=REPO_ID, filename="train_100.jsonl", repo_type="dataset"
)
BENCH_DATA_PATH = hf_hub_download(
    repo_id=REPO_ID, filename="test_50.jsonl", repo_type="dataset"
)

print(f"Train: {TRAIN_DATA_PATH}")
print(f"Bench: {BENCH_DATA_PATH}")

In [ ]:
def load_jsonl(path):
    """Load a JSONL file into a list of Python dicts (one dict per line)."""
    with open(path, "r") as f:
        return [json.loads(line) for line in f if line.strip()]

train_raw = load_jsonl(TRAIN_DATA_PATH)
bench_raw = load_jsonl(BENCH_DATA_PATH)

from collections import Counter
print("=" * 55)
print(f"  TRAIN  — {len(train_raw)} teacher reasoning traces")
print(f"  Labels:  {dict(Counter(r['phenotype'] for r in train_raw))}")
print()
print(f"  TEST   — {len(bench_raw)} held-out benchmark isolates")
print(f"  Labels:  {dict(Counter(r['phenotype'] for r in bench_raw))}")
print("=" * 55)
print()
print("--- Question preview (first 300 chars) ---")
print(train_raw[0]["question"][:300])
print()
print("--- Answer preview (first 300 chars) ---")
print(train_raw[0]["answer"][:300])
print("...")
print(train_raw[0]["answer"][-120:])

---
## 3. Configuration

In [ ]:
# ── LoRA (paper: rank 16, lr 1e-4, 10 epochs) ────────────────────────────────
LOAD_IN_4BIT  = True    # Load weights as 4-bit integers → 4× memory saving
LORA_RANK     = 16      # Rank r of adapter matrices A and B
LORA_ALPHA    = 16      # Scaling factor (conventionally set equal to rank)
LEARNING_RATE = 1e-4    # Step size for gradient descent (paper value)
NUM_EPOCHS    = 10      # How many full passes through the training data

# ── Inference ─────────────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 16384  # Maximum tokens the model can generate (paper: 16,384)

# ── Batch ─────────────────────────────────────────────────────────────────────
BATCH_SIZE = 1          # Examples processed per GPU step (limited by VRAM on T4)
GRAD_ACCUM = 4          # Gradient accumulation steps → effective batch = 1×4 = 4

# ── Output paths ──────────────────────────────────────────────────────────────
OUTPUT_DIR   = "/content/bactereason-checkpoints"
ADAPTER_PATH = "/content/bactereason-lora"

print(f"Model:            {BASE_MODEL}")
print(f"Max seq length:   {MAX_SEQ_LENGTH} tokens")
print(f"LoRA rank:        {LORA_RANK}  (new params per layer: {2 * 4096 * LORA_RANK:,})")
print(f"Effective batch:  {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"Epochs:           {NUM_EPOCHS}")
print(f"Train examples:   {len(train_raw)}")

---
## 4. Load Model & Attach LoRA Adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

# Step 4a: Load pre-trained base model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=torch.float16,    # FP16 for T4 compatibility
    offload_buffers=True,
    device_map='auto',
)

# Step 4b: Freeze original weights; inject LoRA adapters into Attention + FFN layers
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # Self-Attention projections
        "gate_proj", "up_proj", "down_proj",        # Feed-Forward Network
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Recompute activations to save VRAM
)

# Should show ~1-2% trainable parameters (the LoRA A and B matrices only)
model.print_trainable_parameters()

---
## 5. Prepare Training Dataset

In [ ]:
from datasets import Dataset

def format_to_chat_text(example):
    """
    Wrap (question, answer) in the model's chat template.
    This produces the exact text format the model was pre-trained with.
    add_generation_prompt=False: include the assistant's answer in the text (SFT mode).
    """
    messages = [
        {"role": "user",      "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,   # SFT mode: include answer
    )
    return {"text": text}

# Convert list of dicts → HuggingFace Dataset object (required by SFTTrainer)
train_dataset = Dataset.from_list(train_raw)
train_dataset = train_dataset.map(format_to_chat_text)

print(f"Dataset: {len(train_dataset)} formatted examples")
print(f"\n--- Formatted text (first 500 chars) ---")
print(train_dataset[0]["text"][:500])

---
## 6. Distillation via LoRA Fine-Tuning (SFT)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,               # Process each trace independently
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,   # Effective batch = 4
        num_train_epochs=NUM_EPOCHS,              # 10 full passes (paper)
        learning_rate=LEARNING_RATE,              # 1e-4 (paper)
        lr_scheduler_type="cosine",               # Smooth LR decay
        warmup_ratio=0.05,                        # 5% linear warmup
        fp16=not torch.cuda.is_bf16_supported(),  # FP16 on T4
        bf16=torch.cuda.is_bf16_supported(),       # BF16 on A100
        logging_steps=10,                         # Print loss every 10 steps
        save_strategy="epoch",                    # Checkpoint after each epoch
        seed=42,                                  # Reproducibility
        report_to="none",
    ),
)

In [ ]:
print(f"Training {BASE_MODEL} on {len(train_dataset)} teacher reasoning traces...")
print(f"Each trace = a step-by-step molecular reasoning written by Claude Opus 4.5 + TogoMCP")
print()
train_result = trainer.train()

mins = train_result.metrics["train_runtime"] / 60
print(f"\nDone in {mins:.1f} min")
print(f"  Total steps: {train_result.global_step}")
print(f"  Final loss:  {train_result.training_loss:.4f}")
print()
print("Note: loss decreasing over epochs means the student is learning the teacher's reasoning style.")

In [ ]:
# Save only the LoRA adapter weights (A and B matrices) — much smaller than the full model

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to: {ADAPTER_PATH}/")
print("(Only ~50-200 MB, not the full ~20 GB model)")

# Persist to Drive:
# !cp -r {ADAPTER_PATH} /content/drive/MyDrive/bactereason-lora

---
## 7. Inference on Benchmark

In [ ]:
# If resuming after a runtime restart, uncomment to reload the fine-tuned adapter:
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=ADAPTER_PATH, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True,
# )

FastLanguageModel.for_inference(model)
print("Model ready for inference.")

In [ ]:
import re

def extract_prediction(text):
    """
    Parse the model's generated reasoning trace to extract the final verdict.

    The model outputs something like:
      Plain:  'Therefore, SAMN14755011 is susceptible to ceftriaxone.'
      Bold:   '**Therefore, SAMN04014910 is resistant to cefotaxime.**'

    Strategy: scan sentences from the END (verdict always appears last),
    return the first sentence with exactly one of the two class keywords.
    'missing' means we could not find a clear verdict within the token limit.
    """
    cleaned   = re.sub(r'\*\*', '', text)   # strip **bold** markdown
    sentences = re.split(r'(?<=[.!?])\s+', cleaned.strip())
    for sent in reversed(sentences):
        s = sent.lower()
        has_s = "susceptible" in s
        has_r = "resistant"   in s
        if has_s and not has_r:
            return "susceptible"
        if has_r and not has_s:
            return "resistant"
    return "missing"


def generate_one(question):
    """
    Run the fine-tuned model on one benchmark question.

    Key difference from training: add_generation_prompt=True
      → do NOT include an assistant answer in the input
      → ask the model to GENERATE a new answer

    do_sample=False → greedy decoding: always pick the highest-probability token.
    This makes the output deterministic and reproducible (paper setting).
    """
    messages   = [{"role": "user", "content": question}]
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,   # Inference mode: ask model to generate
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():   # No gradient needed at inference → saves memory
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,   # Up to 16,384 new tokens
            do_sample=False,                 # Greedy decoding: deterministic
        )

    # Decode only the newly generated tokens (strip the input prompt from output)
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
print(f"Benchmark: {len(bench_raw)} held-out isolates")
print(f"  Susceptible: {sum(1 for r in bench_raw if r['phenotype'] == 'susceptible')}")
print(f"  Resistant:   {sum(1 for r in bench_raw if r['phenotype'] == 'resistant')}")
print()

results = []
for i, ex in enumerate(bench_raw):
    output = generate_one(ex["question"])    # use 'question' as model input
    pred   = extract_prediction(output)
    results.append({
        "index":        i,
        "biosample":    ex["biosample"],
        "antibiotic":   ex["antibiotic"],
        "species":      ex.get("species", ""),
        "ground_truth": ex["phenotype"],     # 'phenotype' is the ground-truth label
        "prediction":   pred,
        "reasoning":    output,
    })
    if (i + 1) % 10 == 0:
        n_ok    = sum(1 for r in results if r["ground_truth"] == r["prediction"])
        n_valid = sum(1 for r in results if r["prediction"] != "missing")
        acc_str = f"{n_ok/n_valid:.1%}" if n_valid > 0 else "N/A"
        print(f"  [{i+1:2d}/{len(bench_raw)}]  acc so far: {acc_str}  "
              f"(missing: {len(results) - n_valid})")

print("\nInference complete.")

---
## 8. Evaluation
### Accuracy and Error Rate
### Confusion Matrix

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

df        = pd.DataFrame(results)
n_missing = (df["prediction"] == "missing").sum()
df_valid  = df[df["prediction"] != "missing"].copy()

print(f"Total isolates: {len(df)}")
print(f"Valid preds:    {len(df_valid)}  (model produced a clear susceptible/resistant label)")
print(f"Missing:        {n_missing}   (no clear label within {MAX_NEW_TOKENS}-token limit)")

In [ ]:
if len(df_valid) > 0:
    acc        = accuracy_score(df_valid["ground_truth"], df_valid["prediction"])
    error_rate = 1 - acc
    print(f"Accuracy:   {acc:.1%}")
    print(f"Error rate: {error_rate:.1%}")
    print(f"(Paper: BacteReason 18% on A100 with 200 isolates)")
    print()

    labels = ["susceptible", "resistant"]
    cm = confusion_matrix(
        df_valid["ground_truth"], df_valid["prediction"], labels=labels
    )
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(
        cm, display_labels=["Susceptible", "Resistant"]
    ).plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(
        f"{BASE_MODEL.split('/')[-1]}\nError rate: {error_rate:.1%}  |  n={len(df_valid)}"
    )
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Ground truth label")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)
    plt.show()

    # Show the breakdown
    tn = cm[0][0]; fp = cm[0][1]; fn = cm[1][0]; tp = cm[1][1]
    print(f"True Negatives  (susceptible correctly predicted): {tn}")
    print(f"False Positives (susceptible predicted as resistant): {fp}")
    print(f"False Negatives (resistant predicted as susceptible): {fn}  ← clinical risk!")
    print(f"True Positives  (resistant correctly predicted): {tp}")
else:
    print("No valid predictions — check model output format.")

In [ ]:
df.to_json("benchmark_results.jsonl", orient="records", lines=True)
print("Results saved: benchmark_results.jsonl")

# from google.colab import files
# files.download("benchmark_results.jsonl")
# files.download("confusion_matrix.png")

---
## 9. Inspect Reasoning Traces

In [ ]:
def show_example(idx):
    row   = df_valid.iloc[idx]
    match = "✅ CORRECT" if row["ground_truth"] == row["prediction"] else "❌ WRONG"
    print("=" * 70)
    print(f"{match}   |   Truth: {row['ground_truth']}   Pred: {row['prediction']}")
    print(f"BioSample: {row['biosample']}  |  Species: {row['species']}")
    print(f"Antibiotic: {row['antibiotic']}")
    print("=" * 70)
    trace = row["reasoning"]
    if len(trace) > 3000:
        print(trace[:1500])
        print(f"\n  [...{len(trace) - 3000} chars of reasoning omitted...]\n")
        print(trace[-1500:])
    else:
        print(trace)

correct = df_valid[df_valid["ground_truth"] == df_valid["prediction"]]
wrong   = df_valid[df_valid["ground_truth"] != df_valid["prediction"]]

if len(correct) > 0:
    print(">>> CORRECT EXAMPLE — check for sound mechanistic reasoning <<<\n")
    show_example(correct.index[0])
if len(wrong) > 0:
    print("\n\n>>> INCORRECT EXAMPLE — look for confusion or hallucination <<<\n")
    show_example(wrong.index[0])